## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [ ]:
load_dotenv(override=True)

In [ ]:
# Constants 

MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [ ]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [ ]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [ ]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [ ]:
WebSearchPlan.model_json_schema()

In [ ]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [ ]:

result = await Runner.run(planner_agent, task)
result.final_output

## Agent 3: The Writer Agent

In [ ]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## Agent 4: The email agent

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [ ]:
send_email_tool.params_json_schema

In [ ]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [ ]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Showtime!

In [ ]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>

### Write a research agent with custom **serpapi** websearch tool

In [53]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, trace, Runner, OpenAIChatCompletionsModel, function_tool, ModelSettings
from pydantic import BaseModel, Field
from messenger import send_email
from IPython.display import display, Markdown
import serpapi
import asyncio

load_dotenv(override=True)
NUM_OF_SEARCHES = 5

gemini_api_key = os.getenv("GOOGLE_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
serpapi_api_key = os.getenv("SERPAPI_API_KEY")

gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_base_url = "https://openrouter.ai/api/v1"

gemini_client = AsyncOpenAI(api_key=gemini_api_key, base_url=gemini_base_url)
openrouter_client = AsyncOpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)

gemini_model = OpenAIChatCompletionsModel(openai_client=gemini_client, model="gemini-2.5-flash-lite")
openrouter_model = OpenAIChatCompletionsModel(openai_client=openrouter_client, model="cohere/north-mini-code:free")

Structured outputs

In [54]:
class WebSearchParams(BaseModel):
    query: str = Field(description="The search query string")

class QUERY_CLARIFY(BaseModel):
    query: str = Field(description="The user query")
    clarified: list = Field(description="Answered clarifing questions by user to clarify the query search domain.")

class WEB_SEARCH_ITEM(BaseModel):
    reason: str = Field(description="Your reasoning for searching this term for the quey.")
    query: str = Field(description="your search term for the search query.")

class WEB_SEARCH_LIST(BaseModel):
    searches: list[WEB_SEARCH_ITEM] = Field(description="list of web searchs to perform to best answer the query")

class REPORT(BaseModel):
    summary: str = Field(description="The 2-3 sentence summary of the final findings.")
    markdown: str = Field(description="The final report.")
    further_question : str = Field(description="suggested topic to research further.")

Tools

In [55]:

@function_tool
async def web_search(params: WebSearchParams) -> dict:
    """Search the web for current information"""

    search_params = {
        "q": params.query,
        "api_key": serpapi_api_key,
        "engine": "google",
    }

    client = serpapi.Client(api_key=serpapi_api_key)
    response = client.search(search_params)

    organic_results = response.get("organic_results", []) if response else []
    results = [{"title": r.get("title"), "link": r.get("link"), "snippet": r.get("snippet")} for r in organic_results[:3]]

    formatted = "\n\n".join(
        f"Result {i + 1}:\nTitle: {r['title']}\nSnippet: {r['snippet']}\nLink: {r['link']}" for i, r in enumerate(results)
        )

    return {
        "result": (
            f"Search results:\n\n{formatted}\n\n"
            "Based on the above results, answer the query appropriately."
        )
    }


@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)
    return "Email sent successfully"

Agents

In [80]:
clarifing_instruction = """
You are a research assistant. Given a user query, you clarify the user query with some questions and request from the user to answer the questions. 
With considering answered questions related to the query, create a supplementary query from a combination of the user query and the clarifying answers. 
"""

planner_instruction = """
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {NUM_OF_SEARCHES} terms to query for.
"""
search_instruction = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
writer_istruction = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""

emailer_instruction = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

settings = ModelSettings(tool_choice="required")

clarifing_agent = Agent("clarifing agent", instructions=clarifing_instruction, model=openrouter_model, output_type=QUERY_CLARIFY)
planner_agent = Agent("planner agent", instructions=planner_instruction, model=openrouter_model, output_type=WEB_SEARCH_LIST)
search_agent = Agent("search agent", instructions=search_instruction, model=openrouter_model, tools=[web_search], model_settings=settings)
report_agent = Agent("reporter agent", instructions=writer_istruction, model=openrouter_model, output_type=REPORT)
email_agent = Agent("email agent", instructions=emailer_instruction, model=openrouter_model, tools=[send_email_tool], model_settings=settings)

Orchestrate via code

In [ ]:

async def run_search(query: str):
    query_result = await Runner.run(clarifing_agent, query)
    search_query = f"The user query is: {query_result.final_output.query} \n and clarification regarding the query is: {query_result.final_output.clarified}"
    print(f"Search query: /n {search_query}")

    searches_result = await Runner.run(planner_agent, search_query)
    
    searches = searches_result.final_output.searches
    print(f"perform {len(searches)} searches . . .  ")
    tasks = [search(search_item) for search_item in searches]

    results = await asyncio.gather(*tasks)
    print("End searching")
    return results

async def search(query_item):
    query = f"The query is: {query_item.query}, and the reason is: {query_item.reason}"
    result = await Runner.run(search_agent, query)
    return result.final_output

async def write_report(query: str, results: list):
    print("Prepare report ...")
    write_report_query = f"The query is: {query}, and summarized response is: {results}"
    results = await Runner.run(report_agent, write_report_query)
    return results.final_output

async def send_report_mail(report: REPORT):
    print("Sending report as en email")
    result = await Runner.run(email_agent, report.markdown)
    print(result.final_output)

In [ ]:
Query = "what are the best AI agents frameworks?"
with trace("Research trace"):
    search_results = await run_search(Query)
    report = await write_report(Query, search_results)
    await send_report_mail(report)

Orchestrate with LLM tools

In [81]:
clarifing_tool = clarifing_agent.as_tool(tool_name="clarifing_tool", tool_description="Use this tool to clarify the user query.")
planner_tool = planner_agent.as_tool(tool_name="planner_tool", tool_description="Use this tool to break down the user query into a set of web searches.")
search_tool = search_agent.as_tool(tool_name="search_tool", tool_description="Use this tool to search a web for the given term.")
report_tool = report_agent.as_tool(tool_name="report_tool", tool_description="Use this tool prepare a report of the search findings.")
email_tool = email_agent.as_tool(tool_name="email_tool", tool_description="Use this tool to send an email")

tools = [clarifing_tool, planner_tool, search_tool, report_tool, email_tool]
settings = ModelSettings(tool_choice="required")

manager_instruction = """
You are a research assistant manager. Given a user query, you should clarify the query with some questions and find a supplamantry query, breakdown that query into multiple web searches to find best answer for the user, prepare a report of findings using your tools, and send that report as an email for the user.
Use "clarifing_tool" for clarifing the user query and make a supplamantary query.
Use "planner_tool" to breakdown the supplamantary query into a set of web searches.
Use "search_tool" to search a query related term from web.
USe "report_tool" to create a report of findings from web searches.
Use "email_tool" to send a finding report as an email to the user.
Finally, generate the final response based on the tools and present it to the user.
"""

research_manager_agent = Agent("research manager agent", instructions=manager_instruction, model=openrouter_model, tools=tools, model_settings=settings)

In [ ]:
Query = "what are the best AI agents frameworks?"
with trace("Deep research"):
    result = await Runner.run(research_manager_agent, Query)
    result.final_output

Orchestrate with LLM handoffs

In [ ]:

research_instruction = """
You are a research assistant manager. Given a user query, you should clarify the query with some questions and find a supplamantry query, breakdown that query into multiple web searches to find best answer for the user, prepare a report of findings using your tools, and send that report as an email for the user.
handoff to the clarifing agent for clarifing the user query and make a supplamantary query.
handoff to the planner agent to breakdown the supplamantary query into a set of web searches.
handoff to the search agent to search a query related term from web.
handoff to the report agent to create a report of findings from web searches.
handoff to the email agent to send a finding report as an email to the user.
"""

research_agent = Agent("research manager agent", instructions=manager_instruction, model=openrouter_model, tools=tools, model_settings=settings)

In [ ]:
Query = "what are the best AI agents frameworks?"
with trace("Deep research"):
    result = await Runner.run(research_agent, Query)
    result.final_output